# DATA LAMA (OKT-DES)

In [5]:
import pandas as pd
import numpy as np
import warnings

# --- KONFIGURASI ---
warnings.filterwarnings('ignore')
nama_file_input = 'Data Sampah.xlsx'
nama_file_output = 'Laporan_Analisa_Waste_Management_OktDes.xlsx'
start_date = '2025-10-01'
end_date = '2025-12-31'
cutoff_date = pd.Timestamp('2025-12-27')
HARGA_EKSTERNAL_PER_RIT = 350000
KAPASITAS_TRUK = 2500

print("=== MEMPROSES DATA (REVISI DATA EKSTERNAL) ===")

try:
    # ---------------------------------------------------------
    # 1. PROSES DATA INTERNAL
    # ---------------------------------------------------------
    df_int = pd.read_excel(nama_file_input, sheet_name='Internal')
    df_int['Tgl kegiatan'] = pd.to_datetime(df_int['Tgl kegiatan'], errors='coerce')
    df_int = df_int[(df_int['Tgl kegiatan'] >= start_date) & (df_int['Tgl kegiatan'] <= end_date)].copy()
    
    # Cleaning & Calculation
    cols_biaya = {
        'Nominal\nParkir': 'Parkir', 'Nominal\nTol': 'Tol', 
        'Nominal\nTambal ban /\nTambah angin': 'Tambal Ban', 
        'Biaya Retribusi\nMasuk TPS': 'Retribusi', 'Debit': 'Debit'
    }
    df_int.rename(columns=cols_biaya, inplace=True)
    list_komponen = list(cols_biaya.values())
    
    for col in list_komponen:
        if col in df_int.columns:
            df_int[col] = pd.to_numeric(df_int[col], errors='coerce').fillna(0)
    
    df_int['Total_Biaya_Ops'] = df_int[list_komponen].sum(axis=1)
    
    # Logic Seleksi Baris Cost Internal
    cond_1 = (df_int['Tgl kegiatan'] <= cutoff_date) & (df_int['BS'].notna()) & (df_int['VO'].notna())
    cond_2 = (df_int['Tgl kegiatan'] > cutoff_date)
    mask_valid = cond_1 | cond_2
    
    df_valid_int = df_int[mask_valid].copy()
    total_ops_internal = df_valid_int['Total_Biaya_Ops'].sum()
    breakdown_ops = df_valid_int[list_komponen].sum().reset_index()
    breakdown_ops.columns = ['Komponen Operasional', 'Total Biaya (Rp)']
    
    # Berat Internal
    df_int['Brt Bersih'] = pd.to_numeric(df_int['Brt Bersih'], errors='coerce').fillna(0)
    total_weight_internal = df_int['Brt Bersih'].sum()

    # ---------------------------------------------------------
    # 2. BBM & MAINTENANCE
    # ---------------------------------------------------------
    df_bbm = pd.read_excel(nama_file_input, sheet_name='Data BBM')
    df_bbm['Tanggal'] = pd.to_datetime(df_bbm['Tanggal'], errors='coerce')
    df_bbm = df_bbm[(df_bbm['Tanggal'] >= start_date) & (df_bbm['Tanggal'] <= end_date)].copy()
    total_bbm = df_bbm['Total'].sum()
    
    df_maint = pd.read_excel(nama_file_input, sheet_name='Data Maintenance')
    df_maint['TANGGAL'] = pd.to_datetime(df_maint['TANGGAL'], errors='coerce')
    df_maint = df_maint[df_maint['TANGGAL'].dt.month.isin([10, 11, 12])].copy()
    total_maint = df_maint['JUMLAH_'].sum()
    
    total_cost_internal_final = total_ops_internal + total_bbm + total_maint

    # ---------------------------------------------------------
    # 3. EKSTERNAL (REVISI FILTER)
    # ---------------------------------------------------------
    try:
        df_raw = pd.read_excel(nama_file_input, sheet_name='Eksternal', header=None)
        sheet_ext = 'Eksternal'
    except:
        df_raw = pd.read_excel(nama_file_input, sheet_name='External', header=None)
        sheet_ext = 'External'
        
    header_idx = df_raw[df_raw.apply(lambda row: row.astype(str).str.contains('TANGGAL', case=False).any(), axis=1)].index[0]
    df_ext = pd.read_excel(nama_file_input, sheet_name=sheet_ext, header=header_idx)
    df_ext['TANGGAL'] = pd.to_datetime(df_ext['TANGGAL'], errors='coerce')
    
    # FILTER PENTING: Pastikan hanya range tanggal yang diminta
    df_ext = df_ext[(df_ext['TANGGAL'] >= start_date) & (df_ext['TANGGAL'] <= end_date)].copy()
    
    total_cost_ext = len(df_ext) * HARGA_EKSTERNAL_PER_RIT
    total_weight_ext = df_ext['VOLUME'].sum() * 300

    # ---------------------------------------------------------
    # 4. ANALISA LANJUTAN
    # ---------------------------------------------------------
    # A. Utilitas Internal
    df_int['Load_Factor_%'] = (df_int['Brt Bersih'] / KAPASITAS_TRUK) * 100
    df_int['Kategori_Utilitas'] = pd.cut(
        df_int['Load_Factor_%'], 
        bins=[-1, 50, 75, 90, 999], 
        labels=['Inefisien (<50%)', 'Kurang (50-75%)', 'Baik (75-90%)', 'Optimal (>90%)']
    )
    summary_util_int = df_int.groupby('Kategori_Utilitas')['Brt Bersih'].count().reset_index(name='Jumlah Trip')

    # B. Efisiensi Eksternal (Revisi Angka)
    df_ext['Status_Efisiensi'] = np.where(df_ext['VOLUME'] < 4, 'Boros (< 4m3)', 'Optimal (>= 4m3)')
    summary_util_ext = df_ext.groupby('Status_Efisiensi')['VOLUME'].agg(['count', 'mean']).reset_index()
    
    rit_boros = len(df_ext[df_ext['Status_Efisiensi'] == 'Boros (< 4m3)'])
    potensi_hemat = (rit_boros / 2) * HARGA_EKSTERNAL_PER_RIT
    insight_ext = pd.DataFrame({'Info': ['Potensi Hemat (Konsolidasi)'], 'Nilai': [potensi_hemat]})

    # C. Pareto
    col_loc = [c for c in df_int.columns if 'Lokasi' in c][0]
    df_int['Lokasi_Clean'] = df_int[col_loc].astype(str).str.upper().str.strip()
    pareto_int = df_int.groupby('Lokasi_Clean')['Brt Bersih'].sum().sort_values(ascending=False).head(10).reset_index()

    # ---------------------------------------------------------
    # 5. EXPORT EXCEL
    # ---------------------------------------------------------
    summary_main = pd.DataFrame({
        'Kategori': ['Vendor INTERNAL', 'Vendor EKSTERNAL'],
        'Total Biaya (Rp)': [total_cost_internal_final, total_cost_ext],
        'Total Berat (Kg)': [total_weight_internal, total_weight_ext],
        'Cost per Kg (Rp)': [total_cost_internal_final/total_weight_internal, total_cost_ext/total_weight_ext],
        'Jumlah Trip/Rit': [len(df_int), len(df_ext)]
    })
    
    breakdown_final = breakdown_ops.copy()
    breakdown_final = pd.concat([breakdown_final, pd.DataFrame({
        'Komponen Operasional': ['Bahan Bakar (BBM)', 'Maintenance (Service)', 'TOTAL INTERNAL'], 
        'Total Biaya (Rp)': [total_bbm, total_maint, total_cost_internal_final]
    })], ignore_index=True)

    print(f"Menulis file ke: {nama_file_output}...")
    with pd.ExcelWriter(nama_file_output) as writer:
        summary_main.to_excel(writer, sheet_name='Ringkasan Executive', startrow=1, index=False)
        breakdown_final.to_excel(writer, sheet_name='Ringkasan Executive', startrow=7, index=False)
        
        df_int.to_excel(writer, sheet_name='Detail Internal', index=False)
        df_bbm.to_excel(writer, sheet_name='Detail BBM', index=False)
        df_maint.to_excel(writer, sheet_name='Detail Maintenance', index=False)
        df_ext.to_excel(writer, sheet_name='Detail Eksternal', index=False)
        
        sh = 'Analisa Lanjutan'
        summary_util_int.to_excel(writer, sheet_name=sh, startrow=1, index=False)
        summary_util_ext.to_excel(writer, sheet_name=sh, startrow=9, index=False)
        insight_ext.to_excel(writer, sheet_name=sh, startrow=14, index=False)
        pareto_int.to_excel(writer, sheet_name=sh, startrow=19, index=False)

    print(f"SUKSES! File {nama_file_output} terupdate dengan data 16 rit boros.")

except Exception as e:
    print(f"Error: {e}")

=== MEMPROSES DATA (REVISI DATA EKSTERNAL) ===
Menulis file ke: Laporan_Analisa_Waste_Management_OktDes.xlsx...
SUKSES! File Laporan_Analisa_Waste_Management_OktDes.xlsx terupdate dengan data 16 rit boros.


# DATA LAMA (OKT-NOV)

In [1]:
import pandas as pd
import numpy as np
import warnings

# --- KONFIGURASI ---
warnings.filterwarnings('ignore')
nama_file_input = 'Data Sampah.xlsx'
nama_file_output = 'Laporan_Analisa_Waste_Management_OktNov.xlsx'

# Update Periode: Oktober s/d November
start_date = '2025-10-01'
end_date = '2025-11-30' 

cutoff_date = pd.Timestamp('2025-12-27')
HARGA_EKSTERNAL_PER_RIT = 350000

print("=== MEMPROSES DATA ANALISA (PERBAIKAN COST: NO DEBIT) ===")

try:
    # ---------------------------------------------------------
    # 1. LOAD DATA INTERNAL (RAW)
    # ---------------------------------------------------------
    df_raw_int = pd.read_excel(nama_file_input, sheet_name='Internal')
    df_raw_int['Tgl kegiatan'] = pd.to_datetime(df_raw_int['Tgl kegiatan'], errors='coerce')
    
    # Cleaning Kolom Biaya pada Data Raw
    cols_biaya = {
        'Nominal\nParkir': 'Parkir', 
        'Nominal\nTol': 'Tol', 
        'Nominal\nTambal ban /\nTambah angin': 'Tambal Ban', 
        'Biaya Retribusi\nMasuk TPS': 'Retribusi', 
        'Debit': 'Debit'
    }
    df_raw_int.rename(columns=cols_biaya, inplace=True)
    
    # REVISI: Tentukan komponen yang DIHITUNG saja (Tanpa Debit)
    list_komponen_hitung = ['Parkir', 'Tol', 'Tambal Ban', 'Retribusi']
    
    # Cleaning numeric (Loop semua kolom biaya biar rapi datanya, termasuk Debit)
    for col in cols_biaya.values():
        if col in df_raw_int.columns:
            df_raw_int[col] = pd.to_numeric(df_raw_int[col], errors='coerce').fillna(0)
    
    # REVISI: Hitung Total Hanya dari komponen hitung (Parkir, Tol, Tambal Ban, Retribusi)
    valid_cols_hitung = [c for c in list_komponen_hitung if c in df_raw_int.columns]
    df_raw_int['Total_Biaya_Ops'] = df_raw_int[valid_cols_hitung].sum(axis=1)
    
    df_raw_int['Brt Bersih'] = pd.to_numeric(df_raw_int['Brt Bersih'], errors='coerce').fillna(0)

    # ---------------------------------------------------------
    # 2. HITUNG BIAYA 30 SEPTEMBER (PENGURANG)
    # ---------------------------------------------------------
    # Kita cari biaya tanggal 30 September untuk dikurangkan nantinya
    # (Total_Biaya_Ops di sini sudah bersih dari Debit)
    cost_sept_30 = df_raw_int[df_raw_int['Tgl kegiatan'] == '2025-09-30']['Total_Biaya_Ops'].sum()
    print(f"Info: Biaya 30 September yang akan dikurangkan: Rp {cost_sept_30:,.0f}")

    # ---------------------------------------------------------
    # 3. FILTER PERIODE (OKT - NOV)
    # ---------------------------------------------------------
    df_int = df_raw_int[(df_raw_int['Tgl kegiatan'] >= start_date) & (df_raw_int['Tgl kegiatan'] <= end_date)].copy()
    
    # Logic Seleksi Baris (Validasi Cost)
    cond_1 = (df_int['Tgl kegiatan'] <= cutoff_date) & (df_int['BS'].notna()) & (df_int['VO'].notna())
    cond_2 = (df_int['Tgl kegiatan'] > cutoff_date)
    mask_valid = cond_1 | cond_2
    
    # Tandai Status
    df_int.loc[mask_valid, 'Status_Hitung'] = 'DIPAKAI'
    df_int.loc[~mask_valid, 'Status_Hitung'] = 'INFO SAJA'
    
    # ---------------------------------------------------------
    # 4. HITUNG TOTAL BIAYA INTERNAL (FINAL)
    # ---------------------------------------------------------
    df_valid_int = df_int[mask_valid].copy()
    
    # Total Ops Awal (Sum Oktober-November)
    total_ops_gross = df_valid_int['Total_Biaya_Ops'].sum()
    
    # Total Ops Net (Dikurangi 30 Sept)
    total_ops_net = total_ops_gross - cost_sept_30
    
    # Breakdown Komponen (REVISI: Hanya menjumlahkan komponen hitung, tanpa Debit)
    breakdown_ops = df_valid_int[valid_cols_hitung].sum().reset_index()
    breakdown_ops.columns = ['Komponen Operasional', 'Total Biaya (Rp)']

    # Total Berat
    total_weight_internal = df_int['Brt Bersih'].sum()

    # ---------------------------------------------------------
    # 5. DATA BBM & MAINTENANCE
    # ---------------------------------------------------------
    df_bbm = pd.read_excel(nama_file_input, sheet_name='Data BBM')
    df_bbm['Tanggal'] = pd.to_datetime(df_bbm['Tanggal'], errors='coerce')
    df_bbm = df_bbm[(df_bbm['Tanggal'] >= start_date) & (df_bbm['Tanggal'] <= end_date)].copy()
    total_bbm = df_bbm['Total'].sum()
    
    df_maint = pd.read_excel(nama_file_input, sheet_name='Data Maintenance')
    df_maint['TANGGAL'] = pd.to_datetime(df_maint['TANGGAL'], errors='coerce')
    # Filter Maintenance Bulan 10 & 11
    df_maint = df_maint[df_maint['TANGGAL'].dt.month.isin([10, 11])].copy()
    total_maint = df_maint['JUMLAH_'].sum()
    
    # TOTAL INTERNAL FINAL
    total_cost_internal_final = total_ops_net + total_bbm + total_maint

    # ---------------------------------------------------------
    # 6. DATA EKSTERNAL
    # ---------------------------------------------------------
    try:
        df_raw_ext = pd.read_excel(nama_file_input, sheet_name='Eksternal', header=None)
        sheet_ext_name = 'Eksternal'
    except:
        df_raw_ext = pd.read_excel(nama_file_input, sheet_name='External', header=None)
        sheet_ext_name = 'External'
        
    header_idx = df_raw_ext[df_raw_ext.apply(lambda row: row.astype(str).str.contains('TANGGAL', case=False).any(), axis=1)].index[0]
    df_ext = pd.read_excel(nama_file_input, sheet_name=sheet_ext_name, header=header_idx)
    df_ext['TANGGAL'] = pd.to_datetime(df_ext['TANGGAL'], errors='coerce')
    
    # Filter Periode
    df_ext = df_ext[(df_ext['TANGGAL'] >= start_date) & (df_ext['TANGGAL'] <= end_date)].copy()
    
    total_cost_ext = len(df_ext) * HARGA_EKSTERNAL_PER_RIT
    total_weight_ext = df_ext['VOLUME'].sum() * 300

    # =========================================================
    # 7. ANALISA LANJUTAN (UTILITAS DINAMIS)
    # =========================================================
    
    # A. Tentukan Kapasitas Maksimal Truk (Dinamis dari Data)
    MAX_CAPACITY = df_int['Brt Bersih'].max()
    if MAX_CAPACITY == 0: MAX_CAPACITY = 2500 # Fallback
    print(f"Info: Kapasitas Maksimal Truk yang digunakan: {MAX_CAPACITY} Kg")

    # B. Hitung Utilitas
    df_int['Load_Factor_%'] = (df_int['Brt Bersih'] / MAX_CAPACITY) * 100
    df_int['Kategori_Utilitas'] = pd.cut(
        df_int['Load_Factor_%'], 
        bins=[-1, 50, 75, 90, 999], 
        labels=['Inefisien (<50%)', 'Kurang (50-75%)', 'Baik (75-90%)', 'Optimal (>90%)']
    )
    summary_util_int = df_int.groupby('Kategori_Utilitas')['Brt Bersih'].count().reset_index(name='Jumlah Trip')

    # C. Efisiensi Eksternal
    df_ext['Status_Efisiensi'] = np.where(df_ext['VOLUME'] < 4, 'Boros (< 4m3)', 'Optimal (>= 4m3)')
    summary_util_ext = df_ext.groupby('Status_Efisiensi')['VOLUME'].agg(['count', 'mean']).reset_index()
    
    rit_boros = len(df_ext[df_ext['Status_Efisiensi'] == 'Boros (< 4m3)'])
    potensi_hemat = (rit_boros / 2) * HARGA_EKSTERNAL_PER_RIT
    insight_ext = pd.DataFrame({'Info': ['Potensi Hemat'], 'Nilai': [potensi_hemat]})

    # D. Pareto
    col_loc = [c for c in df_int.columns if 'Lokasi' in c][0]
    df_int['Lokasi_Clean'] = df_int[col_loc].astype(str).str.upper().str.strip()
    pareto_int = df_int.groupby('Lokasi_Clean')['Brt Bersih'].sum().sort_values(ascending=False).head(10).reset_index()

    # ---------------------------------------------------------
    # 8. EXPORT KE EXCEL
    # ---------------------------------------------------------
    summary_main = pd.DataFrame({
        'Kategori': ['Vendor INTERNAL', 'Vendor EKSTERNAL'],
        'Total Biaya (Rp)': [total_cost_internal_final, total_cost_ext],
        'Total Berat (Kg)': [total_weight_internal, total_weight_ext],
        'Cost per Kg (Rp)': [total_cost_internal_final/total_weight_internal, total_cost_ext/total_weight_ext],
        'Jumlah Trip/Rit': [len(df_int), len(df_ext)]
    })
    
    # Siapkan Tabel Rincian Biaya
    breakdown_final = breakdown_ops.copy()
    # Baris Penyesuaian Sept 30
    adj_row = pd.DataFrame({
        'Komponen Operasional': ['KOREKSI (Data 30 Sept)', 'Bahan Bakar (BBM)', 'Maintenance (Service)'], 
        'Total Biaya (Rp)': [-cost_sept_30, total_bbm, total_maint]
    })
    breakdown_final = pd.concat([breakdown_final, adj_row], ignore_index=True)
    
    # Baris Total
    total_row = pd.DataFrame({
        'Komponen Operasional': ['TOTAL INTERNAL FINAL'],
        'Total Biaya (Rp)': [total_cost_internal_final]
    })
    breakdown_final = pd.concat([breakdown_final, total_row], ignore_index=True)

    print(f"Menulis file ke: {nama_file_output}...")
    
    with pd.ExcelWriter(nama_file_output) as writer:
        # Sheet 1: Ringkasan
        pd.DataFrame(["RINGKASAN EXECUTIVE (OKT - NOV 2025)"]).to_excel(writer, sheet_name='Ringkasan Executive', index=False, header=False)
        summary_main.to_excel(writer, sheet_name='Ringkasan Executive', startrow=1, index=False)
        
        pd.DataFrame(["RINCIAN BIAYA INTERNAL (Dengan Koreksi 30 Sept)"]).to_excel(writer, sheet_name='Ringkasan Executive', startrow=7, index=False, header=False)
        breakdown_final.to_excel(writer, sheet_name='Ringkasan Executive', startrow=8, index=False)
        
        # Details
        df_int.to_excel(writer, sheet_name='Detail Internal', index=False)
        df_bbm.to_excel(writer, sheet_name='Detail BBM', index=False)
        df_maint.to_excel(writer, sheet_name='Detail Maintenance', index=False)
        df_ext.to_excel(writer, sheet_name='Detail Eksternal', index=False)
        
        # Analisa Lanjutan
        sh = 'Analisa Lanjutan'
        pd.DataFrame([f"A. UTILITAS INTERNAL (Benchmark Max: {MAX_CAPACITY} Kg)"]).to_excel(writer, sheet_name=sh, index=False, header=False)
        summary_util_int.to_excel(writer, sheet_name=sh, startrow=1, index=False)
        
        pd.DataFrame(["B. EFISIENSI EKSTERNAL"]).to_excel(writer, sheet_name=sh, startrow=8, index=False, header=False)
        summary_util_ext.to_excel(writer, sheet_name=sh, startrow=9, index=False)
        insight_ext.to_excel(writer, sheet_name=sh, startrow=14, index=False)
        
        pd.DataFrame(["C. PARETO SUMBER SAMPAH"]).to_excel(writer, sheet_name=sh, startrow=18, index=False, header=False)
        pareto_int.to_excel(writer, sheet_name=sh, startrow=19, index=False)

    print(f"SUKSES! File '{nama_file_output}' berhasil dibuat.")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

=== MEMPROSES DATA ANALISA (PERBAIKAN COST: NO DEBIT) ===
Info: Biaya 30 September yang akan dikurangkan: Rp 56,500
Info: Kapasitas Maksimal Truk yang digunakan: 2040.0 Kg
Menulis file ke: Laporan_Analisa_Waste_Management_OktNov.xlsx...
SUKSES! File 'Laporan_Analisa_Waste_Management_OktNov.xlsx' berhasil dibuat.
